# Notebook 5: Data Preparation and Feature Engineering
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Handle missing values systematically
- Detect and treat outliers
- Encode categorical variables correctly
- Scale numerical features
- Engineer new features from existing ones
- Build a reusable scikit-learn pipeline

## 1. Why Data Preparation Matters

> "Data scientists spend 80% of their time cleaning data and 20% complaining about cleaning data."

A model is only as good as the data it learns from. Dirty data → bad predictions, even with a perfect algorithm.

### The Data Preparation Pipeline
```
Raw Data
  ↓ Inspection & profiling
  ↓ Handle missing values
  ↓ Handle outliers
  ↓ Encode categoricals
  ↓ Scale / normalise
  ↓ Feature engineering
Clean Feature Matrix
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder,
    OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

np.random.seed(42)

# Synthetic messy dataset
def make_messy_dataset(n=200):
    df = pd.DataFrame({
        'age'    : np.random.randint(18, 70, n).astype(float),
        'income' : np.random.normal(50000, 15000, n),
        'score'  : np.random.normal(70, 15, n),
        'city'   : np.random.choice(['London','NYC','Paris','Berlin'], n),
        'edu'    : np.random.choice(['High School','Bachelor','Master','PhD'], n),
        'target' : np.random.randint(0, 2, n),
    })
    # Introduce missing values
    df.loc[df.sample(20).index, 'age'] = np.nan
    df.loc[df.sample(15).index, 'income'] = np.nan
    df.loc[df.sample(5).index, 'city'] = np.nan
    # Introduce outliers
    df.loc[[0,1,2], 'income'] = [500000, -5000, 1000000]
    return df

df = make_messy_dataset()
print(df.shape)
df.head()

## 2. Inspecting and Profiling Data

In [ ]:
print('=== Data Profile ===')
print(f'Shape  : {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nMissing %:\n{(df.isnull().mean()*100).round(1)}')
print(f'\nDtypes:\n{df.dtypes}')

numerical = df.select_dtypes(include=np.number).columns.tolist()
categorical = df.select_dtypes(exclude=np.number).columns.tolist()
print(f'\nNumerical columns : {numerical}')
print(f'Categorical columns: {categorical}')

## 3. Handling Missing Values

### Strategies
| Strategy | When to use |
|----------|-------------|
| **Drop rows** | Very few rows affected, data is plentiful |
| **Drop columns** | > 50–70% of column is missing |
| **Mean / median fill** | Numerical; median is robust to outliers |
| **Mode fill** | Categorical |
| **KNN imputation** | When missing values have patterns |
| **Model-based** | Complex relationships between features |

In [ ]:
# Mean / median imputation
imp_median = SimpleImputer(strategy='median')
imp_mode   = SimpleImputer(strategy='most_frequent')

df_clean = df.copy()
df_clean[['age','income','score']] = imp_median.fit_transform(df_clean[['age','income','score']])
df_clean['city'] = imp_mode.fit_transform(df_clean[['city']]).ravel()

print('Missing values after imputation:')
print(df_clean.isnull().sum())

# KNN imputation (uses neighbouring samples)
knn_imp = KNNImputer(n_neighbors=5)
df_knn = df.copy()
df_knn[['age','income','score']] = knn_imp.fit_transform(df_knn[['age','income','score']])
print('\nKNN imputed income sample:', df_knn['income'].head(3).values.round(0))

## 4. Detecting and Treating Outliers

### Methods
- **Z-score**: flag values where |z| > 3
- **IQR (Interquartile Range)**: flag values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR
- **Winsorisation**: cap at a percentile instead of removing

In [ ]:
income = df_clean['income']

# Z-score method
z_scores = (income - income.mean()) / income.std()
print(f'Outliers (|z|>3): {(np.abs(z_scores) > 3).sum()}')

# IQR method
Q1, Q3 = income.quantile(0.25), income.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outlier_mask = (income < lower) | (income > upper)
print(f'Outliers (IQR)  : {outlier_mask.sum()}')
print(f'Outlier values  : {income[outlier_mask].values}')

# Winsorise (cap at 1st and 99th percentile)
df_clean['income_winsor'] = np.clip(df_clean['income'],
                                     df_clean['income'].quantile(0.01),
                                     df_clean['income'].quantile(0.99))
print(f'\nIncome max before: {df_clean["income"].max():,.0f}')
print(f'Income max after : {df_clean["income_winsor"].max():,.0f}')

## 5. Encoding Categorical Variables

| Encoding | When |
|----------|------|
| **Label Encoding** | Ordinal categories (Low < Medium < High) |
| **One-Hot Encoding** | Nominal categories (no order); watch dummy-variable trap |
| **Target Encoding** | High-cardinality columns in advanced workflows |

In [ ]:
# One-Hot Encoding
city_ohe = pd.get_dummies(df_clean['city'], prefix='city', drop_first=True)
print('One-hot encoded cities:\n', city_ohe.head())

# Ordinal Encoding for education
edu_order = [['High School','Bachelor','Master','PhD']]
ordenc = OrdinalEncoder(categories=edu_order)
df_clean['edu_enc'] = ordenc.fit_transform(df_clean[['edu']])
print('\nEducation ordinal mapping:')
for level, enc in zip(edu_order[0], range(4)):
    print(f'  {level} → {enc}')

## 6. Feature Scaling

Many algorithms (SVM, KNN, logistic regression, neural networks) are sensitive to feature scale.

| Scaler | Formula | Best when |
|--------|---------|----------|
| **StandardScaler** | (x − μ) / σ | Features roughly normal |
| **MinMaxScaler** | (x − min) / (max − min) | Bounded output needed [0,1] |
| **RobustScaler** | (x − median) / IQR | Many outliers |

In [ ]:
from sklearn.preprocessing import RobustScaler

feat = df_clean[['age','income_winsor','score']]

std_scaled   = StandardScaler().fit_transform(feat)
minmax_scaled = MinMaxScaler().fit_transform(feat)
robust_scaled = RobustScaler().fit_transform(feat)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes,
    [std_scaled, minmax_scaled, robust_scaled],
    ['Standard (z-score)', 'Min-Max [0,1]', 'Robust (median/IQR)']):
    ax.boxplot(data, labels=['age','income','score'])
    ax.set_title(title)
plt.suptitle('Scaling Comparison')
plt.tight_layout()
plt.show()

## 7. Feature Engineering

Creating **new features** from existing ones often boosts model performance more than tuning hyperparameters.

In [ ]:
df_fe = df_clean.copy()

# Ratio feature
df_fe['income_per_age'] = df_fe['income_winsor'] / df_fe['age']

# Binning
df_fe['age_group'] = pd.cut(df_fe['age'], bins=[0,25,40,60,100],
                             labels=['Young','Mid','Senior','Elder'])

# Log transform (reduces skewness)
df_fe['log_income'] = np.log1p(df_fe['income_winsor'])

# Polynomial features
df_fe['score_sq'] = df_fe['score'] ** 2
df_fe['age_score'] = df_fe['age'] * df_fe['score']

print('New features:')
print(df_fe[['income_per_age','age_group','log_income','score_sq','age_score']].head())

## 8. Building a scikit-learn Pipeline

A `Pipeline` chains transformers and estimators — ensuring the same transformations are applied to train and test data consistently, preventing **data leakage**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df_clean[['age','income_winsor','score','city','edu_enc']]
y = df_clean['target']

X_ohe = pd.get_dummies(X, columns=['city'], drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_ohe, y, test_size=0.2, random_state=42)

num_cols = ['age','income_winsor','score','edu_enc']
ohe_cols = [c for c in X_ohe.columns if c.startswith('city')]

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('ohe', 'passthrough',   ohe_cols),
])

pipe = Pipeline([
    ('preprocessor', pre),
    ('model', LogisticRegression(max_iter=500)),
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(f'Pipeline accuracy: {accuracy_score(y_test, pred):.3f}')

## Exercises

1. Using the Titanic dataset (`seaborn.load_dataset('titanic')`), build a complete preprocessing pipeline: impute missing values, encode categoricals, scale numericals.
2. Create a feature `title` extracted from the passenger's name (Mr, Mrs, Miss, etc.) and encode it.
3. Compare model accuracy before and after feature engineering on any dataset.
4. What happens if you scale training data and then fit a new scaler on test data separately? (Run the experiment and explain why this is wrong.)

## Reflection Questions
- What is **data leakage** and how does a `Pipeline` prevent it?
- Why is `StandardScaler` inappropriate for features with heavy outliers?
- Why should you use `drop_first=True` in one-hot encoding?

---
**Next →** `06_linear_regression.ipynb`